# EGM vs NEGM: gamma_c sweep recovery

Compare how EGM(FUES) and NEGM(FUES) recover all 5 parameters from selfgen data as the true gamma_c varies. Each sweep point generates data at a fixed gamma_c, then estimates all 5 params including gamma_c. Results shown for both males and females.

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*IProgress.*')

import json, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath('../../..')
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

GADI = os.path.expanduser('~/gadi/g/data/tp66/results/durables2_0/estimation')

# True parameter values (from calibration.yaml)
THETA_TRUE = {
    'alpha': 0.7, 'beta': 0.945, 'gamma_h': 1.5, 'tau': 0.12,
    # gamma_c varies per sweep point — plotted as 45-degree line
}

print(f'Results: {GADI}')

## 1. Load sweep results

In [ ]:
def load_sweep(spec_name):
    """Load the latest sweep_summary CSV, or build from individual point summaries."""
    spec_dir = os.path.join(GADI, spec_name)
    if not os.path.isdir(spec_dir):
        return None

    # Try sweep_summary first
    csvs = sorted([f for f in os.listdir(spec_dir) if f.startswith('sweep_summary')])
    if csvs:
        return pd.read_csv(os.path.join(spec_dir, csvs[-1]))

    # Fallback: load individual point summaries
    rows = []
    for point_dir in sorted(os.listdir(spec_dir)):
        point_path = os.path.join(spec_dir, point_dir)
        if not os.path.isdir(point_path) or '=' not in point_dir:
            continue
        run_dirs = sorted([d for d in os.listdir(point_path)
                           if os.path.isfile(os.path.join(point_path, d, 'summary.json'))])
        if not run_dirs:
            continue
        with open(os.path.join(point_path, run_dirs[-1], 'summary.json')) as f:
            s = json.load(f)
        gc = float(point_dir.split('=')[1])
        rows.append({'gamma_c': gc, **s['theta_best'],
                     'objective': s['objective'], 'converged': s['converged'],
                     'n_iter': s['n_iter']})
    return pd.DataFrame(rows) if rows else None

# Load all four sweep variants
sweeps = {}
for label, spec in [
    ('EGM females',  'selfgen_sweep_gamma_c_egm'),
    ('NEGM females', 'selfgen_sweep_gamma_c_negm'),
    ('EGM males',    'selfgen_sweep_gamma_c_egm'),    # same spec name in separable_males
    ('NEGM males',   'selfgen_sweep_gamma_c_negm'),
]:
    df = load_sweep(spec)
    if df is not None and len(df) > 0:
        sweeps[label] = df
        print(f'{label}: {len(df)} points, loss range [{df["objective"].min():.4f}, {df["objective"].max():.4f}]')
    else:
        print(f'{label}: no data')

## 2. Parameter recovery across gamma_c

In [ ]:
styles = {
    'EGM females':  dict(color='#2166ac', linestyle='-', marker='o', markersize=5, linewidth=2),
    'NEGM females': dict(color='#b2182b', linestyle='-', marker='s', markersize=5, linewidth=2),
    'EGM males':    dict(color='#2166ac', linestyle='--', marker='^', markersize=5, linewidth=2, alpha=0.5),
    'NEGM males':   dict(color='#b2182b', linestyle='--', marker='v', markersize=5, linewidth=2, alpha=0.5),
}

free_params = ['gamma_c', 'alpha', 'beta', 'gamma_h', 'tau']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for i, p in enumerate(free_params):
    ax = axes[i]

    # True value reference
    if p == 'gamma_c':
        ax.plot([1, 9], [1, 9], 'k:', linewidth=1, label='True')
    elif p in THETA_TRUE:
        ax.axhline(THETA_TRUE[p], color='k', linestyle=':', linewidth=1, label='True')

    for label, df in sweeps.items():
        if 'gamma_c' in df.columns and p in df.columns:
            ax.plot(df['gamma_c'], df[p], label=label, **styles[label])

    ax.set_xlabel(r'True $\gamma_c$')
    ax.set_ylabel(f'Estimated ${p}$')
    ax.set_title(f'${p}$', fontsize=12)
    ax.legend(fontsize=7, frameon=False)
    ax.grid(True, alpha=0.2)

# Loss panel
ax = axes[5]
for label, df in sweeps.items():
    if 'gamma_c' in df.columns and 'objective' in df.columns:
        ax.plot(df['gamma_c'], df['objective'], label=label, **styles[label])
ax.set_xlabel(r'True $\gamma_c$')
ax.set_ylabel('SMM loss')
ax.set_yscale('log')
ax.set_title('Loss', fontsize=12)
ax.legend(fontsize=7, frameon=False)
ax.grid(True, alpha=0.2)

fig.suptitle(r'Selfgen recovery across true $\gamma_c$: EGM vs NEGM, males vs females',
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 3. Recovery error table

In [ ]:
# Mean absolute percentage error across sweep points for each method/gender
from IPython.display import Markdown

rows = []
for label, df in sweeps.items():
    for _, row in df.iterrows():
        true_gc = row['gamma_c']
        for p in ['alpha', 'beta', 'gamma_c', 'gamma_h', 'tau']:
            true_val = true_gc if p == 'gamma_c' else THETA_TRUE[p]
            est_val = row[p]
            pct_err = abs(est_val - true_val) / true_val * 100
            rows.append({
                'method': label, 'true_gamma_c': true_gc,
                'param': p, 'true': true_val, 'est': est_val,
                'pct_err': pct_err,
            })

err_df = pd.DataFrame(rows)

# Summary: mean % error by method and param
summary = err_df.pivot_table(index='method', columns='param',
                             values='pct_err', aggfunc='mean')
summary = summary[['gamma_c', 'alpha', 'beta', 'gamma_h', 'tau']]

print('Mean absolute percentage error (%) across all gamma_c sweep points:')
print()
print(summary.round(3).to_string())

## 4. Convergence speed

Number of CE iterations to converge at each gamma_c sweep point.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, df in sweeps.items():
    if 'gamma_c' in df.columns and 'n_iter' in df.columns:
        ax.plot(df['gamma_c'], df['n_iter'], label=label, **styles[label])

ax.set_xlabel(r'True $\gamma_c$')
ax.set_ylabel('CE iterations to convergence')
ax.set_title('Convergence speed: EGM vs NEGM')
ax.legend(fontsize=9, frameon=False)
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()

---

*Source: `examples/durables2_0/` — Dobrescu and Shanker (2026), Application 2*